In [1]:
!git clone https://github.com/jasonfghx/Medical-voice-recognition_processing_system.git

Cloning into 'Medical-voice-recognition_processing_system'...
remote: Enumerating objects: 2710, done.
remote: Counting objects: 100% (304/304), done.
remote: Compressing objects: 100% (274/274), done.
remote: Total 2710 (delta 83), reused 30 (delta 30), pack-reused 2406 (from 2)
Receiving objects: 100% (2710/2710), 975.36 MiB | 23.75 MiB/s, done.
Resolving deltas: 100% (369/369), done.
Updating files: 100% (2378/2378), done.


In [ ]:
BRANCH = 'main'
!python -m pip install git+https://github.com/NVIDIA/NeMo.git@$BRANCH#egg=nemo_toolkit[asr]

DEPRECATION: git+https://github.com/NVIDIA/NeMo.git@main#egg=nemo_toolkit[asr] contains an egg fragment with a non-PEP 508 name pip 25.0 will enforce this behaviour change. A possible replacement is to use the req @ url syntax, and remove the egg fragment. Discussion can be found at https://github.com/pypa/pip/issues/11617
  Cloning https://github.com/NVIDIA/NeMo.git (to revision main) to /tmp/pip-install-4156_iw9/nemo-toolkit_2b74fdb8ffa84bae81d578cbba354d65
  Running command git clone --filter=blob:none --quiet https://github.com/NVIDIA/NeMo.git /tmp/pip-install-4156_iw9/nemo-toolkit_2b74fdb8ffa84bae81d578cbba354d65
  Resolved https://github.com/NVIDIA/NeMo.git to commit 29ea0d411814ded50e59f35794e1928d03e9ab3b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 6

In [ ]:
from glob import glob
import random
import json
Cello=glob("/content/Medical-voice-recognition_processing_system/cough/Negative/*.wav")
temp_Cello=[]

for i in Cello:
  json_temp={'audio_filepath':i,
        'duration':2,
        'label':'N',
        'text':'_',
        'offset':0}
  temp_Cello.append(json_temp)

temp_P=[]

Positive=glob("/content/Medical-voice-recognition_processing_system/cough/Positive/*.wav")
for i in Positive*10:
  json_temp={'audio_filepath':i,
        'duration':2,
        'label':'P',
        'text':'_',
        'offset':0}
  temp_P.append(json_temp)

train_temp=temp_Cello[:int(len(temp_Cello)*0.8)]+temp_P[:int(len(temp_P)*0.8)]

test_temp=temp_Cello[int(len(temp_Cello)*0.8):int(len(temp_Cello)*0.9)+1]+temp_P[int(len(temp_P)*0.8):int(len(temp_P)*0.9)+1]

self_temp=[{'audio_filepath':'/content/Medical-voice-recognition_processing_system/cough/Negative/544_Negative_male_27_cough.wav',
      'duration':2,'label':'N','text':'_','offset':0}]

with open('/content/train.json', 'w') as f:
  for i in train_temp:
    json.dump(i, f)
    f.write('\n')
with open('/content/test.json', 'w') as f:
  for i in test_temp:
    json.dump(i, f)
    f.write('\n')





In [ ]:
import nemo
(nemo).__version__

'2.3.0rc0'

In [ ]:
import nemo.collections.asr as nemo_asr
import numpy as np
import os
from omegaconf import OmegaConf
import os, time
import nemo



In [ ]:
train_dataset = '/content/train.json'
val_dataset = '/content/test.json'
test_dataset ='/content/test.json'


In [ ]:
!mkdir configs

In [ ]:
MODEL_CONFIG = "marblenet_3x2x64.yaml"
import os
if not os.path.exists(f"configs/{MODEL_CONFIG}"):
  !wget -P configs/ "https://raw.githubusercontent.com/NVIDIA/NeMo/main/examples/asr/conf/marblenet/{MODEL_CONFIG}"

--2025-02-24 05:51:38--  https://raw.githubusercontent.com/NVIDIA/NeMo/main/examples/asr/conf/marblenet/marblenet_3x2x64.yaml
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4395 (4.3K) [text/plain]
Saving to: ‘configs/marblenet_3x2x64.yaml’

marblenet_3x2x64.ya 100%[===================>]   4.29K  --.-KB/s    in 0s      

2025-02-24 05:51:38 (70.6 MB/s) - ‘configs/marblenet_3x2x64.yaml’ saved [4395/4395]



In [ ]:
config_path = f"configs/{MODEL_CONFIG}"
config = OmegaConf.load(config_path)
config = OmegaConf.to_container(config, resolve=True)
config

In [ ]:
config_path = f"configs/{MODEL_CONFIG}"
config = OmegaConf.load(config_path)
config = OmegaConf.to_container(config, resolve=True)
config = OmegaConf.create(config)

print(OmegaConf.to_yaml(config))
# https://omegaconf.readthedocs.io/en/2.1_branch/usage.html

\

In [ ]:
# labels = config.model.labels
# sample_rate = config.model.sample_rate

In [ ]:
# config.model.sample_rate=1000000000000

In [ ]:
print(OmegaConf.to_yaml(config.model))

In [ ]:
config.model.labels=['N','P']
config.model.train_ds.labels=['N','P']
config.model.validation_ds.labels=['N','P']
config.model.test_ds.labels=['N','P']
config.model.train_ds.manifest_filepath = train_dataset
config.model.validation_ds.manifest_filepath = val_dataset
config.model.test_ds.manifest_filepath = test_dataset

In [ ]:
import torch
import lightning.pytorch as pl

In [ ]:
accelerator = 'gpu' if torch.cuda.is_available() else 'cpu'
config.trainer.devices = 1
config.trainer.accelerator = accelerator

# Reduces maximum number of epochs to 5 for quick demonstration
config.trainer.max_epochs = 2

# Remove distributed training flags
config.trainer.strategy = 'auto'

In [ ]:
trainer = pl.Trainer(**config.trainer)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.utilities.rank_zero:`Trainer(val_check_interval=1.0)` was configured so validation will run at the end of the training epoch..


In [ ]:
from nemo.utils.exp_manager import exp_manager
exp_dir = exp_manager(trainer, config.get("exp_manager", None))

[NeMo I 2025-02-24 05:53:10 nemo_logging:393] ExpManager schema
[NeMo I 2025-02-24 05:53:10 nemo_logging:393] {'explicit_log_dir': None, 'exp_dir': None, 'name': None, 'version': None, 'use_datetime_version': True, 'resume_if_exists': False, 'resume_past_end': False, 'resume_ignore_no_checkpoint': False, 'resume_from_checkpoint': None, 'create_tensorboard_logger': True, 'summary_writer_kwargs': None, 'create_wandb_logger': False, 'wandb_logger_kwargs': None, 'create_mlflow_logger': False, 'mlflow_logger_kwargs': {'experiment_name': None, 'tracking_uri': None, 'tags': None, 'save_dir': './mlruns', 'prefix': '', 'artifact_location': None, 'run_id': None, 'log_model': False}, 'create_dllogger_logger': False, 'dllogger_logger_kwargs': {'verbose': False, 'stdout': False, 'json_file': './dllogger.json'}, 'create_clearml_logger': False, 'clearml_logger_kwargs': {'project': None, 'task': None, 'connect_pytorch': False, 'model_name': None, 'tags': None, 'log_model': False, 'log_cfg': False, 'lo

In [ ]:
exp_dir = str(exp_dir)
exp_dir

In [ ]:
vad_model = nemo_asr.models.EncDecClassificationModel(cfg=config.model, trainer=trainer)


[NeMo I 2025-02-24 05:53:12 nemo_logging:393] Filtered duration for loading collection is  0.00 hours.
[NeMo I 2025-02-24 05:53:12 nemo_logging:393] Dataset successfully loaded with 1341 items and total duration provided from manifest is  0.74 hours.
[NeMo I 2025-02-24 05:53:12 nemo_logging:393] # 1341 files loaded accounting to # 2 labels
[NeMo I 2025-02-24 05:53:12 nemo_logging:393] Filtered duration for loading collection is  0.00 hours.
[NeMo I 2025-02-24 05:53:12 nemo_logging:393] Dataset successfully loaded with 170 items and total duration provided from manifest is  0.09 hours.
[NeMo I 2025-02-24 05:53:12 nemo_logging:393] # 170 files loaded accounting to # 2 labels
[NeMo I 2025-02-24 05:53:12 nemo_logging:393] Filtered duration for loading collection is  0.00 hours.
[NeMo I 2025-02-24 05:53:12 nemo_logging:393] Dataset successfully loaded with 170 items and total duration provided from manifest is  0.09 hours.
[NeMo I 2025-02-24 05:53:12 nemo_logging:393] # 170 files loaded acc

In [ ]:
trainer.fit(vad_model)
trainer.test(vad_model, ckpt_path=None)

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


[NeMo I 2025-02-24 05:53:27 nemo_logging:393] Optimizer config = SGD (
    Parameter Group 0
        dampening: 0
        differentiable: False
        foreach: None
        fused: None
        lr: 0.01
        maximize: False
        momentum: 0.9
        nesterov: False
        weight_decay: 0.001
    )
[NeMo I 2025-02-24 05:53:27 nemo_logging:393] Scheduler "<nemo.core.optim.lr_scheduler.PolynomialHoldDecayAnnealing object at 0x7dd104f155d0>" 
    will be used during training (effective maximum steps = 22) - 
    Parameters : 
    (power: 2.0
    warmup_ratio: 0.05
    hold_ratio: 0.45
    min_lr: 0.001
    last_epoch: -1
    max_steps: 22
    )


INFO: 
  | Name              | Type                         | Params | Mode 
---------------------------------------------------------------------------
0 | spec_augmentation | SpectrogramAugmentation      | 0      | train
1 | preprocessor      | AudioToMFCCPreprocessor      | 0      | train
2 | encoder           | ConvASREncoder               | 88.9 K | train
3 | decoder           | ConvASRDecoderClassification | 258    | train
4 | loss              | CrossEntropyLoss             | 0      | train
5 | _accuracy         | TopKClassificationAccuracy   | 0      | train
---------------------------------------------------------------------------
89.2 K    Trainable params
0         Non-trainable params
89.2 K    Total params
0.357     Total estimated model params size (MB)
103       Modules in train mode
0         Modules in eval mode
INFO:lightning.pytorch.callbacks.model_summary:
  | Name              | Type                         | Params | Mode 
----------------------------------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:Epoch 1, global step 22: 'val_loss' reached 0.71658 (best 0.71658), saving model to '/content/nemo_experiments/MarbleNet-3x2x64/2025-02-24_05-51-57/checkpoints/MarbleNet-3x2x64--val_loss=0.7166-epoch=1.ckpt' as top 3
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=2` reached.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│     test_epoch_top@1      │    0.5235294103622437     │
│         test_loss         │    0.7165824174880981     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.7165824174880981, 'test_epoch_top@1': 0.5235294103622437}]

In [ ]:
vad_model.setup_test_data(config.model.train_ds)
test_dl = vad_model._train_dl
@torch.no_grad()
def extract_logits(model, dataloader):
    logits_buffer = []
    label_buffer = []

    # Follow the above definition of the test_step
    for batch in dataloader:
        audio_signal, audio_signal_len, labels, labels_len = batch
        logits = model(input_signal=audio_signal, input_signal_length=audio_signal_len)

        logits_buffer.append(logits)
        label_buffer.append(labels)
        print(".", end='')
    print()

    print("Finished extracting logits !")
    logits = torch.cat(logits_buffer, 0)
    labels = torch.cat(label_buffer, 0)
    return logits, labels
cpu_model = vad_model.cpu()
cpu_model.eval()
logits, labels = extract_logits(cpu_model, test_dl)

[NeMo I 2025-02-24 05:54:35 nemo_logging:393] Filtered duration for loading collection is  0.00 hours.
[NeMo I 2025-02-24 05:54:35 nemo_logging:393] Dataset successfully loaded with 1341 items and total duration provided from manifest is  0.74 hours.
[NeMo I 2025-02-24 05:54:35 nemo_logging:393] # 1341 files loaded accounting to # 2 labels
...........
Finished extracting logits !


In [ ]:
acc = cpu_model._accuracy(logits=logits, labels=labels)
print(f"Accuracy : {float(acc[0]*100)} %")

Accuracy : 52.796424865722656 %


In [ ]:
import json
self_temp=[{'audio_filepath':'/content/Medical-voice-recognition_processing_system/cough/Positive/745_Positive_male_24_cough.wav',#改這邊
      'duration':2,'label':'P','text':'_','offset':0} ,
           {'audio_filepath':'/content/Medical-voice-recognition_processing_system/cough/Negative/1007_Negative_male_20_cough.wav',#改這邊
      'duration':2,'label':'N','text':'_','offset':0}]
      #
with open('/content/self_test.json', 'w') as f:
  for i in self_temp:
    json.dump(i, f)
    f.write('\n')
test_dataset='/content/self_test.json'
config.model.test_ds.manifest_filepath = test_dataset

In [ ]:
vad_model.setup_test_data(config.model.test_ds)
test_dl = vad_model._test_dl

[NeMo I 2025-02-24 05:55:07 nemo_logging:393] Filtered duration for loading collection is  0.00 hours.
[NeMo I 2025-02-24 05:55:07 nemo_logging:393] Dataset successfully loaded with 2 items and total duration provided from manifest is  0.00 hours.
[NeMo I 2025-02-24 05:55:07 nemo_logging:393] # 2 files loaded accounting to # 2 labels


In [ ]:
# vad_model.setup_test_data(config.model.test_ds)
# test_dl = vad_model._test_dl
@torch.no_grad()
def extract_logits(model, dataloader):
    logits_buffer = []
    label_buffer = []

    # Follow the above definition of the test_step
    for batch in dataloader:
        audio_signal, audio_signal_len, labels, labels_len = batch
        logits = model(input_signal=audio_signal, input_signal_length=audio_signal_len)

        logits_buffer.append(logits)
        label_buffer.append(labels)
        print(".", end='')
    print()

    print("Finished extracting logits !")
    logits = torch.cat(logits_buffer, 0)
    labels = torch.cat(label_buffer, 0)
    return logits, labels
cpu_model = vad_model.cpu()
cpu_model.eval()
logits, labels = extract_logits(cpu_model, test_dl)
# acc = cpu_model._accuracy(logits=logits, labels=labels)
# print(f"Accuracy : {float(acc[0]*100)} %")

.
Finished extracting logits !


In [ ]:
temp={0:'陰性',1:'陽性'}
for i in logits.numpy():
  print(temp[np.argmax(i)])

陰性
陰性
